# 402 — Pruebas de ablación espectral (Corrección #20 del jurado)

Verifica empíricamente que el CNN-2D aprendió la **firma espectral del estrés**
y no la estructura espacial de las parcelas etiquetadas.

**Prueba 1 — Permutación espectral dentro del parche**  
Permuta aleatoriamente los 63 canales de cada parche 5×5. Preserva la estructura
espacial del vecindario pero destruye el contenido espectral. Si el modelo dependiera
del espectro, el rendimiento debe colapsar.

**Prueba 2 — Solo píxel central (sin contexto espacial)**  
Zeriza todos los píxeles vecinos del parche 5×5; conserva únicamente el espectro
del píxel central. Equivale a una evaluación espectral pura. El PR-AUC resultante
es comparable al del CNN-1D real (0.83); la diferencia con el CNN-2D completo (0.96)
cuantifica la ganancia del contexto espacial local legítimo.

**Resultados obtenidos (PC A, 2026-04-30):**
- Prueba 1: PR-AUC=0.7633, ROC-AUC=0.5056 (colapso a aleatorio)
- Prueba 2: PR-AUC=0.8149, ROC-AUC=0.5819 (≈ CNN-1D real)

**Archivos usados:**
- `data/interim/masked_reflectance.zarr`
- `data/interim/bands_selected_by_segment.csv`
- `data/processed/splits/by_plot_split_id_binary.tif`
- `data/raw/labels_export.gpkg`
- `models/cnn2d_final_model_weights.pt` + `models/cnn2d_final_model_info.json`
- `models/robust_scaler.pkl`

## Librerías

In [1]:
import os
import sys

package_path = os.path.abspath('.').split(os.sep + 'notebooks')[0]
if package_path not in sys.path:
    sys.path.append(package_path)

%load_ext autoreload
%autoreload 2

In [2]:
import json
import joblib
import warnings
import time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score

from spectralcrop.utils.path_manager import PathManager

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

PM = PathManager()

def _resolve(path):
    if isinstance(path, list):
        return [p for p in path if 'spectralcrop' not in p][0]
    return path

RAW_PATH       = _resolve(PM.get_abs_path_folder('raw'))
PROCESSED_PATH = _resolve(PM.get_abs_path_folder('processed'))
INTERIM_PATH   = _resolve(PM.get_abs_path_folder('interim'))
MODELS_PATH    = _resolve(PM.get_abs_path_folder('models'))
REPORTS_PATH   = _resolve(PM.get_abs_path_folder('reports'))
FIGURES_DIR    = os.path.join(REPORTS_PATH, 'figures', 'category_D')
os.makedirs(FIGURES_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RNG    = np.random.default_rng(42)

print(f'device={device}')
print(f'FIGURES_DIR={FIGURES_DIR}')

device=cuda
FIGURES_DIR=c:\Users\mario\Documentos\personal_projects\thesis\reports\figures\category_D


## 1. Cargar modelo CNN-2D

In [3]:
class SpectralSpatialCNN2D(nn.Module):
    def __init__(self, n_channels=63, n_classes=2, dropout=0.3, kernel_size=3):
        super().__init__()
        pad = kernel_size // 2
        self.conv1 = nn.Conv2d(n_channels, 32, kernel_size=kernel_size, padding=pad)
        self.bn1   = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=kernel_size, padding=pad)
        self.bn2   = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=kernel_size, padding=pad)
        self.bn3   = nn.BatchNorm2d(128)
        self.pool  = nn.AdaptiveAvgPool2d((2, 2))
        self.fc1   = nn.Linear(128 * 2 * 2, 128)
        self.drop  = nn.Dropout(dropout)
        self.fc2   = nn.Linear(128, n_classes)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.drop(x)
        return self.fc2(x)


with open(os.path.join(MODELS_PATH, 'cnn2d_final_model_info.json')) as f:
    model_info = json.load(f)

model2d = SpectralSpatialCNN2D(
    n_channels=model_info['n_channels'],
    dropout=model_info['dropout'],
    kernel_size=model_info['kernel_size'],
).to(device)
model2d.load_state_dict(
    torch.load(os.path.join(MODELS_PATH, 'cnn2d_final_model_weights.pt'),
               map_location=device, weights_only=False)
)
model2d.eval()

PATCH    = model_info['patch_size']
BEST_THR = model_info['best_thr']
print(f'Modelo listo: {sum(p.numel() for p in model2d.parameters()):,} params')
print(f'patch_size={PATCH}, best_thr={BEST_THR:.4f}')

Modelo listo: 372,994 params
patch_size=5, best_thr=0.3218


## 2. Construcción del test set

Mismo pipeline que notebook 401. Orden de canales: `[NDVI, NDRE, CIgreen, PRI, PSRI, band_1, ..., band_374]`.

In [4]:
t0 = time.time()
print('Cargando feature cube desde masked_reflectance.zarr...')

ds_zarr   = xr.open_zarr(os.path.join(INTERIM_PATH, 'masked_reflectance.zarr'))
bands_df  = pd.read_csv(os.path.join(INTERIM_PATH, 'bands_selected_by_segment.csv'))
sel_bands = bands_df['band_index'].tolist()

VI_NAMES = ['NDVI', 'NDRE', 'CIgreen', 'PRI', 'PSRI']
features_stack = xr.concat(
    [ds_zarr[VI_NAMES].to_array(dim='band'),
     ds_zarr['reflectance'].isel(band=sel_bands)],
    dim='band'
).transpose('y', 'x', 'band')

cube = features_stack.values.astype(np.float32)
H, W, n_feat = cube.shape
print(f'Feature cube: {cube.shape}, elapsed: {time.time()-t0:.1f}s')

scaler      = joblib.load(os.path.join(MODELS_PATH, 'robust_scaler.pkl'))
cube_scaled = scaler.transform(cube.reshape(-1, n_feat)).reshape(H, W, n_feat).astype(np.float32)
print(f'cube_scaled: {cube_scaled.shape}, elapsed: {time.time()-t0:.1f}s')

Cargando feature cube desde masked_reflectance.zarr...
Feature cube: (3660, 3438, 63), elapsed: 19.0s
cube_scaled: (3660, 3438, 63), elapsed: 25.0s


In [5]:
SPLIT_TIF = os.path.join(PROCESSED_PATH, 'splits', 'by_plot_split_id_binary.tif')
with rasterio.open(SPLIT_TIF) as src:
    split_id  = src.read(1)
    transform = src.transform
    crs       = src.crs

split_2d = np.where(split_id == 0, np.nan, split_id.astype(float))

gdf   = gpd.read_file(os.path.join(RAW_PATH, 'labels_export.gpkg'), layer='labels2')
gdf_r = gdf.to_crs(crs)
shapes_bin = [(g, int(v)) for g, v in zip(gdf_r.geometry, gdf_r['binary']) if g]
labels_raw = rasterize(shapes_bin, out_shape=(H, W), transform=transform, fill=-1, dtype=np.int32)
labels_bin = np.where(labels_raw == -1, np.nan, labels_raw.astype(float))

print('Extrayendo parches test (split_id=3)...')
r = PATCH // 2
X_list, y_list = [], []
for i in range(r, H - r):
    for j in range(r, W - r):
        if split_2d[i, j] != 3:
            continue
        if np.isnan(labels_bin[i, j]):
            continue
        patch = cube_scaled[i-r:i+r+1, j-r:j+r+1, :]
        if np.isnan(patch).any():
            continue
        X_list.append(np.transpose(patch, (2, 0, 1)))
        y_list.append(labels_bin[i, j])

X_test = np.stack(X_list)             # (N, 63, 5, 5)
y_test = np.array(y_list, dtype=np.int64)
print(f'Test set: {X_test.shape[0]:,} muestras, elapsed: {time.time()-t0:.1f}s')

Extrayendo parches test (split_id=3)...
Test set: 246,132 muestras, elapsed: 29.8s


## 3. Funciones de inferencia y métricas

In [6]:
def run_inference(X, model, batch_size=512):
    loader = DataLoader(TensorDataset(torch.from_numpy(X)), batch_size=batch_size, shuffle=False)
    probs  = []
    with torch.no_grad():
        for (xb,) in loader:
            xb   = xb.to(device)
            prob = torch.softmax(model(xb), dim=1)[:, 1].cpu().numpy()
            probs.append(prob)
    return np.concatenate(probs)


def report_metrics(label, y_true, y_prob, best_thr):
    pr  = average_precision_score(y_true, y_prob)
    roc = roc_auc_score(y_true, y_prob)
    yp  = (y_prob >= best_thr).astype(int)
    f1  = f1_score(y_true, yp, average='macro')
    print(f'  {label:<40s}  PR-AUC={pr:.4f}  ROC-AUC={roc:.4f}  F1-macro={f1:.4f}')
    return pr, roc, f1


print('Funciones listas.')

Funciones listas.


## 4. Baseline — modelo original (sanity check)

In [7]:
print('Ejecutando inferencia baseline...')
y_prob_orig = run_inference(X_test, model2d)
pr_base, roc_base, f1_base = report_metrics('modelo original', y_test, y_prob_orig, BEST_THR)
assert abs(pr_base - 0.9635) < 0.01, f'Sanity check fallido: {pr_base:.4f}'
print('  Sanity check OK')

Ejecutando inferencia baseline...
  modelo original                           PR-AUC=0.9637  ROC-AUC=0.8902  F1-macro=0.7762
  Sanity check OK


## 5. Prueba 1 — Permutación espectral dentro del parche

Se permutan aleatoriamente los 63 canales de cada parche 5×5. La estructura espacial
del vecindario permanece intacta; el contenido espectral queda destruido. Si el modelo
dependiera del espectro, el rendimiento debe colapsar hacia un clasificador aleatorio.

In [8]:
X_perm = X_test.copy()                               # (N, 63, 5, 5)
for i in range(X_perm.shape[0]):
    perm       = RNG.permutation(n_feat)
    X_perm[i]  = X_perm[i][perm]                     # reordena canales aleatoriamente

y_prob_perm = run_inference(X_perm, model2d)
pr_perm, roc_perm, f1_perm = report_metrics('espectro permutado', y_test, y_prob_perm, BEST_THR)
delta_pr_1 = pr_base - pr_perm
print(f'  Caida de PR-AUC: {delta_pr_1:.4f} ({delta_pr_1/pr_base*100:.1f}% respecto al baseline)')

  espectro permutado                        PR-AUC=0.7633  ROC-AUC=0.5056  F1-macro=0.4982
  Caida de PR-AUC: 0.2004 (20.8% respecto al baseline)


## 6. Prueba 2 — Solo píxel central (sin contexto espacial)

Se zerorizan todos los píxeles vecinos del parche 5×5, conservando únicamente el
espectro del píxel central. Equivale a una evaluación espectral pura. El PR-AUC
resultante es comparable al del CNN-1D real (0.83); la diferencia con el CNN-2D
completo cuantifica la ganancia atribuible al contexto espacial local.

In [9]:
cx = PATCH // 2                                       # índice central = 2 para patch=5
X_center = np.zeros_like(X_test)                      # (N, 63, 5, 5) — todo ceros
X_center[:, :, cx, cx] = X_test[:, :, cx, cx]         # conserva solo el pixel central

y_prob_center = run_inference(X_center, model2d)
pr_center, roc_center, f1_center = report_metrics('solo pixel central', y_test, y_prob_center, BEST_THR)
delta_pr_2 = pr_base - pr_center
print(f'  Caida de PR-AUC: {delta_pr_2:.4f} ({delta_pr_2/pr_base*100:.1f}% respecto al baseline)')

  solo pixel central                        PR-AUC=0.8149  ROC-AUC=0.5819  F1-macro=0.4487
  Caida de PR-AUC: 0.1489 (15.4% respecto al baseline)


## 7. Resumen

In [10]:
results = pd.DataFrame([
    {'condicion': 'Modelo original (baseline)',   'PR_AUC': pr_base,   'ROC_AUC': roc_base,   'F1_macro': f1_base},
    {'condicion': 'Prueba 1: espectro permutado', 'PR_AUC': pr_perm,   'ROC_AUC': roc_perm,   'F1_macro': f1_perm},
    {'condicion': 'Prueba 2: solo pixel central', 'PR_AUC': pr_center, 'ROC_AUC': roc_center, 'F1_macro': f1_center},
])

print(results.to_string(index=False))

results.to_csv(os.path.join(FIGURES_DIR, 'ablation_results.csv'), index=False)
print(f'\nResultados guardados en {FIGURES_DIR}/ablation_results.csv')
print(f'Tiempo total: {time.time()-t0:.1f}s')

                   condicion   PR_AUC  ROC_AUC  F1_macro
  Modelo original (baseline) 0.963707 0.890169  0.776202
Prueba 1: espectro permutado 0.763285 0.505557  0.498231
Prueba 2: solo pixel central 0.814855 0.581921  0.448674

Resultados guardados en c:\Users\mario\Documentos\personal_projects\thesis\reports\figures\category_D/ablation_results.csv
Tiempo total: 43.8s
